# Laboratorio 4 — Enunciado: Temperaturas CRU

## Antes de empezar

El proyecto ya está preparado. Trabajen en parejas y ejecuten los comandos
desde la carpeta de este laboratorio:

```bash
uv sync
uv run jupyter lab
```

El notebook es el espacio para explorar los datos y probar las
transformaciones. El código definitivo debe quedar en `src/meteolab/`.

## Flujo de trabajo

En cada operación seguirán este ciclo:

1. Investigar la operación o el método que necesitan.
2. Probarlo directamente sobre los datos en el notebook.
3. Observar y comprobar el resultado.
4. Trasladar la solución al módulo indicado.
5. Comprobarla con `uv run pytest -m etapaN`.
6. Interpretar los resultados y responder la pregunta de la etapa.

## El dataset CRU

El archivo `data/cru_country_tmp_tidy.csv` contiene temperaturas medias del
conjunto de datos de la *Climatic Research Unit* (CRU). Cada fila representa
un país, un año y un período. El período puede ser un mes (`JAN` a `DEC`),
una estación climática (`DJF`, `MAM`, `JJA`, `SON`) o el promedio anual
(`ANN`).

| Columna | Significado | Tipo esperado |
|---|---|---|
| `country` | nombre del país | `String` |
| `iso_alpha2` | código ISO de dos letras | `String` |
| `iso_alpha3` | código ISO de tres letras | `String` |
| `year` | año de la observación | `Int64` |
| `period` | mes, estación climática o promedio anual | `String` |
| `temperature_c` | temperatura media en grados Celsius | `Float64` |
| `parameter` | indicador medido | `String` |
| `units` | unidad del indicador | `String` |
| `source_file` | archivo de origen | `String` |

El laboratorio se concentrará en las **temperaturas medias mensuales**.
Durante la limpieza descartarán las filas de `DJF`, `MAM`, `JJA`, `SON` y
`ANN`. Desde ese punto, ninguna agregación podrá usar esas observaciones.

```mermaid
flowchart LR
    A["CSV CRU<br/>17 períodos"] --> B["Exploración"]
    B --> C["Lectura y esquema"]
    C --> D["Limpieza<br/>solo JAN–DEC"]
    D --> E["Fechas mensuales"]
    E --> F["Agregaciones y ventanas"]
    F --> G["Pipeline lazy"]
```

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — Polars</strong>
  <strong>Polars</strong> es una librería de Python para trabajar con datos tabulares. Sus estructuras principales son <code>DataFrame</code>, que contiene datos ya materializados, y <code>LazyFrame</code>, que representa una consulta aún no ejecutada. Polars ofrece una API de expresiones para describir transformaciones sobre columnas y permite trabajar en modo <em>eager</em> o <em>lazy</em>.
</div>

## Evaluación

| Etapa | Contenido | Implementación | Análisis y preguntas | Total |
|---|---|---:|---:|---:|
| 1 | Exploración del CSV | 0.2 | 0.4 | 0.6 |
| 2 | Polars e I/O | 0.4 | 0.3 | 0.7 |
| 3 | Esquema y validación | 0.5 | 0.3 | 0.8 |
| 4 | Limpieza y selección mensual | 0.5 | 0.4 | 0.9 |
| 5 | Fechas y agregaciones | 0.7 | 0.4 | 1.1 |
| 6 | Ventanas y anomalías | 0.5 | 0.3 | 0.8 |
| 7 | Pipeline lazy y análisis | 0.3 | 0.8 | 1.1 |
| **Total** | | **3.1** | **2.9** | **6.0** |

## Preparación

In [ ]:
import sys
from pathlib import Path
import plotly.express as px
import polars as pl

RAIZ = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RUTA_DATOS = RAIZ / "data"
RUTA_CSV = RUTA_DATOS / "cru_country_tmp_tidy.csv"

print("Proyecto:", RAIZ)
print("Archivo :", RUTA_CSV)

---
# Etapa 1 — Explorar el archivo (0.6 puntos)

Antes de decidir cómo leer o transformar una tabla, observen sus dimensiones,
sus tipos y sus valores ausentes. Esta etapa no modifica los datos: levanta
evidencia para las decisiones que tomarán después.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — tabla y esquema</strong>
  Una tabla organiza observaciones en filas y variables en columnas. El <strong>esquema</strong> es la lista de columnas junto con sus tipos. En este archivo, el esquema distingue identificadores de texto, años enteros y temperaturas decimales.
</div>

### 1.1 — Leer e inspeccionar (0.1 puntos)

Lean el CSV sin imponer todavía el esquema. En la siguiente etapa justificarán
qué tipos y valores nulos deben declarar explícitamente.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — CSV</strong>
  <strong>CSV</strong> (*Comma-Separated Values*) es un archivo de texto en el que cada fila representa un registro y las columnas se separan mediante un delimitador, normalmente una coma. Es fácil de intercambiar, pero no guarda de forma completa el esquema de la tabla: al leerlo, la librería debe inferir o recibir los tipos de cada columna.
</div>

Investiguen `polars.read_csv` y utilícenla para leer `RUTA_CSV`.

In [ ]:
raw = pl.read_csv(RUTA_CSV)

Inspeccionen las primeras y las últimas diez filas. Luego, muestren diez filas
aleatorias con una semilla fija.

In [ ]:
display(raw.head(10))
display(raw.tail(10))
display(raw.sample(n=10, seed=7202))

Revisen las dimensiones, los nombres de las columnas, el esquema y una vista
compacta de los valores.

In [ ]:
print("Dimensiones:", raw.shape)
print("Columnas:", raw.columns)
print("Esquema:")
print(raw.schema)
raw.glimpse()

Generen un resumen estadístico y cuenten los valores nulos por columna.

In [ ]:
display(raw.describe())
display(raw.null_count())

### 1.2 — Visualizar la distribución (0.1 puntos)

Plotly Express permite construir gráficos interactivos a partir de columnas
tabulares. Exploren la distribución de `temperature_c` sin confundir los
valores ausentes con una temperatura.

In [ ]:
fig = px.histogram(
    raw,
    x="temperature_c",
    nbins=40,
    title="Distribución de las temperaturas CRU",
    labels={"temperature_c": "Temperatura (°C)"},
)
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 1 — Qué muestra el archivo (0.2 puntos)</strong>
  ¿Cuántas filas y columnas tiene el CSV? ¿Qué columnas son identificadores, cuáles representan tiempo y cuál contiene la medición? Expliquen qué información se pierde si `temperature_c` se lee como texto.
</div>

Tiene 408000 filas y 9 columnas
Los identificadores son country, year, period, las que representan tiempo son period y year, y la que contiene la medición es temperature_c. Si se guarda como texto se perderia informacion como estadisticas de media, maximo, minimo etc. </code>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 2 — Escalas temporales (0.2 puntos)</strong>
  El archivo contiene 17 valores distintos en `period`. ¿Por qué no se deben mezclar en un mismo promedio las filas mensuales, estacionales y anuales? Anticipen qué período conservarán durante la limpieza y por qué.
</div>

Porque por ejemplo en Chile la temperatura promedio en verano es muy distinta que en invierno, entonces si queremos hacer un análisis por estaciones no podríamos compararlo con el promedio por estación. Aparte se distorsionaría la métrica.

---
# Etapa 2 — Polars e I/O (0.7 puntos)

En esta etapa pasarán de una lectura exploratoria a una lectura reproducible.
Usarán el CSV como fuente de entrada y compararán la lectura eager con la
lectura lazy.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — eager y lazy</strong>
  En modo <em>eager</em>, una operación se ejecuta cuando se llama y devuelve un <code>DataFrame</code>. En modo <em>lazy</em>, las operaciones construyen un plan y se ejecutan al llamar <code>collect()</code>. <code>scan_csv</code> permite describir una consulta sin cargar de inmediato todo el archivo.
</div>

### 2.1 — Investigar y probar la lectura CSV (0.3 puntos)

Investiguen `schema_overrides` en `read_csv`. Luego, vuelvan a leer el CSV
declarando `year` como `Int64` y `temperature_c` como `Float64`. Comparen el
esquema de esta tabla con el de `raw`.

In [ ]:
lecturas = pl.read_csv(
    RUTA_CSV,
    schema_overrides={
        "year": pl.Int64,
        "temperature_c": pl.Float64
    }
)

In [ ]:
print("Esquema inferido :", raw.schema)
print("Esquema declarado:", lecturas.schema)

Investiguen `scan_csv` y construyan una consulta que filtre algunos países
sin ejecutarla. Comprueben el tipo de objeto y materialicen el resultado con
`collect()`.

In [ ]:
consulta = pl.scan_csv(
        RUTA_CSV).filter(
        pl.col("country").is_in(["Chile", "Argentina", "Brazil"]))

In [ ]:
print(type(consulta))
display(consulta.collect().head())
print(consulta.explain())

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 3 — Elegir la lectura (0.3 puntos)</strong>
  ¿Qué ventaja ofrece declarar `year` y `temperature_c` al leer el CSV? ¿En qué situación tendría sentido usar `scan_csv` en vez de `read_csv`?
</div>

La ventaja es que evita depender de la inferencia automática de tipos y asegura que year se trate como entero y temperature_c como decimal.
scan_csv tiene sentido cuando se trabaja con archivos grandes y se beneficia de lazy.

### 2.2 — Trasladar la lectura CSV al módulo (0.1 puntos)

Después de probar las operaciones, implementen en `src/meteolab/carga.py`:

- `leer_temperaturas(ruta)`, con `schema_overrides`;
- `escanear_temperaturas(ruta)`, que debe devolver un `LazyFrame`;
Comprueben también que `escanear_temperaturas` devuelve un `LazyFrame` y que
la consulta se materializa solo con `collect()`.

In [ ]:
from src.meteolab.carga import (
    escanear_temperaturas,
    leer_temperaturas,
)

lecturas_modulo = leer_temperaturas(RUTA_CSV)
consulta_modulo = escanear_temperaturas(RUTA_CSV)

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Al módulo — <code>src/meteolab/carga.py</code></strong>
  Trasladen las funciones de lectura al módulo. Comprueben con <code>uv run pytest -m etapa2</code> que los tipos, los nulos y el modo lazy se comporten como en el notebook.
</div>

---
# Etapa 3 — Tipos y validación (0.8 puntos)

Leer un archivo no basta: hay que comprobar que los nombres, tipos y valores
permitidos cumplen el contrato del dataset.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — inferir, forzar y validar</strong>
  <strong>Inferir</strong> es dejar que la librería decida un tipo a partir de los valores observados. <strong>Forzar</strong> es declarar el tipo esperado al leer o transformar. <strong>Validar</strong> es comprobar que, además del tipo, los valores cumplen restricciones del dominio.
</div>

### 3.1 — Investigar y declarar el esquema (0.3 puntos)

Investiguen los tipos de Polars y construyan en el notebook un diccionario
con el esquema esperado. Comparen ese diccionario con `lecturas.schema` y
expliquen cualquier diferencia.

In [ ]:
esquema_notebook = esquema_esperado = {
    "country": pl.String,
    "iso_alpha2": pl.String,
    "iso_alpha3": pl.String,
    "year": pl.Int64,
    "period": pl.String,
    "temperature_c": pl.Float64,
    "parameter": pl.String,
    "units": pl.String,
    "source_file": pl.String,
}

In [ ]:
diferencias = {
    columna: (lecturas.schema.get(columna), tipo)
    for columna, tipo in esquema_notebook.items()
    if lecturas.schema.get(columna) != tipo
}
print("Diferencias:", diferencias)

Investiguen Pandera y definan las restricciones que corresponden a este
dataset: años entre 1901 y 2025, períodos permitidos, `Mean Temperature` y
`degrees Celsius`.

In [ ]:
from src.meteolab.constantes import PERIODOS_VALIDOS

In [ ]:
import pandera.polars as pa
validacion_notebook = pa.DataFrameSchema(
    {
        "year": pa.Column(
            int,
            checks=pa.Check.in_range(1901, 2025),
        ),
        "period": pa.Column(
            str,
            checks=pa.Check.isin(PERIODOS_VALIDOS),
        ),
        "statistic": pa.Column(
            str,
            checks=pa.Check.eq("Mean Temperature"),
        ),
        "units": pa.Column(
            str,
            checks=pa.Check.eq("degrees Celsius"),
        ),
    }
)

Los 17 valores de `period` pertenecen al contrato del CSV. Que existan en el
esquema no significa que todos vayan a entrar al análisis: la selección de
períodos mensuales se hará en la limpieza.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 4 — El contrato y el análisis (0.2 puntos)</strong>
  ¿Por qué conviene aceptar `DJF`, `MAM`, `JJA` y `ANN` al validar el archivo, pero excluirlos después durante la limpieza? Relacionen la respuesta con la diferencia entre validar una fuente y definir el universo del análisis.
</div>

Conviene aceptar DJF, MAM, JJA y ANN en la validación porque son valores válidos dentro del contrato del archivo original. Sin embargo, se excluyen durante la limpieza porque el análisis se restringe a períodos mensuales. Validar la fuente comprueba que los datos respeten su formato y valores permitidos, mientras que definir el universo del análisis determina qué observaciones son relevantes para el objetivo del estudio.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 5 — Validar tipos y valores (0.1 puntos)</strong>
  ¿Qué aporta Pandera, además de comparar `lecturas.schema` con el esquema esperado? Mencionen una restricción de valores que Pandera pueda comprobar en este dataset.
</div>

Pandera permite validar no solo los tipos de las columnas, sino también restricciones sobre sus valores. Por ejemplo, puede comprobar que year esté entre 1901 y 2025, o que period pertenezca al conjunto de períodos válidos definido en PERIODOS_VALIDOS.

### 3.2 — Trasladar la validación al módulo (0.2 puntos)

Después de probar el esquema y sus restricciones, implementen en
`src/meteolab/esquema.py`:

- `comparar_esquema`, para informar columnas faltantes y tipos distintos;
- `validar_esquema`, para rechazar un esquema incorrecto;
- `ESQUEMA_TEMPERATURAS`, con Pandera;
- `validar_datos` y `casos_que_fallan`.

In [ ]:
from src.meteolab.esquema import (
    ESQUEMA_TEMPERATURAS,
    casos_que_fallan,
    validar_datos,
    validar_esquema,
)

validar_esquema(lecturas_modulo)
validado = validar_datos(lecturas_modulo)
print("Filas validadas:", validado.height)
print(ESQUEMA_TEMPERATURAS)

In [ ]:
muestra_invalida = lecturas_modulo.with_columns(
    pl.lit("XYZ").alias("period")
)

fallas = casos_que_fallan(muestra_invalida)

---
# Etapa 4 — Limpieza: conservar solo meses (0.9 puntos)

Esta es la decisión central del laboratorio. El archivo mezcla tres escalas
temporales. Desde este punto trabajarán solo con las temperaturas medias de
`JAN` a `DEC`.

In [ ]:
from src.meteolab.constantes import (
    PERIODOS_MENSUALES,
)

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Contrato de limpieza</strong>
  La tabla limpia debe contener únicamente los 12 períodos mensuales. Debe descartar <code>PERIODOS_ESTACIONALES = ("DJF", "MAM", "JJA", "SON")</code> y <code>PERIODO_ANUAL = "ANN"</code>. En este dataset, el nulo conocido pertenece a <code>DJF</code> de 2025 y desaparece al aplicar la selección mensual.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — filtrar filas</strong>
  Filtrar una tabla significa conservar las filas que cumplen una condición. Una condición sobre <code>period</code> define el universo temporal del análisis; no es lo mismo que corregir un valor ni que eliminar una columna.
</div>


### 4.1 — Investigar y limpiar (0.3 puntos)

Investiguen cómo combinar condiciones con `&` y cómo comprobar nulos. Luego,
filtren en el notebook la tabla para conservar solo `PERIODOS_MENSUALES` y
valores disponibles de `temperature_c`. Comprueben las filas y el esquema
resultantes.


In [ ]:
from src.meteolab.constantes import PERIODOS_MENSUALES

limpias = validado.filter(
    pl.col("period").is_in(PERIODOS_MENSUALES)
    & pl.col("temperature_c").is_not_null()
)

In [ ]:
display(limpias.null_count())

In [ ]:
assert set(limpias["period"].unique()) <= {
    "JAN",
    "FEB",
    "MAR",
    "APR",
    "MAY",
    "JUN",
    "JUL",
    "AUG",
    "SEP",
    "OCT",
    "NOV",
    "DEC",
}
assert limpias["temperature_c"].null_count() == 0
print("Filas mensuales limpias:", limpias.height)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 6 — No mezclar escalas (0.1 puntos)</strong>
  ¿Qué problema produciría calcular una media por país usando a la vez meses, estaciones y `ANN`? Expliquen qué filas quedan en la tabla limpia y qué representa cada una.
</div>

No conviene calcular una media por país mezclando meses, estaciones y ANN, porque representan escalas temporales distintas y se estaría contando información agregada junto con información mensual, sesgando el resultado. Después de la limpieza quedan solo filas mensuales (JAN a DEC) con temperature_c disponible. Cada fila representa la temperatura media de un país para un mes y año determinados.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 7 — Comprobar la completitud mensual (0.1 puntos)</strong>
  Después de la limpieza, ¿cómo comprobarían que cada combinación de país y año tiene doce observaciones mensuales? ¿Qué decisión tomarían si faltara un mes?
</div>

Agruparía por país y año y contaría cuántos períodos mensuales distintos existen en cada grupo. Cada combinación país-año debería tener 12 meses. Si faltara alguno, identificaría esas combinaciones y no las trataría como años completos; dependiendo del objetivo, las excluiría o las reportaría como incompletas, en vez de asumir o inventar el valor faltante.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 8 — Imputar con el promedio (0.2 puntos)</strong>
  El nulo conocido pertenece a `DJF` de 2025. Calculen el promedio global de `temperature_c` y supongan que, antes de filtrar los períodos, reemplazan ese valor por dicho promedio. ¿Qué valor se asignaría, qué pasaría con la tabla y qué problema introduciría esa imputación? Expliquen por qué en este laboratorio es preferible descartar la fila.
</div>

Si se reemplazara el nulo de DJF 2025 por el promedio global de temperature_c, se le asignaría un valor que no necesariamente representa ese país, año ni estación. La tabla dejaría de tener ese nulo, pero se introduciría información artificial que podría reducir la variabilidad y distorsionar los análisis. En este laboratorio es preferible descartarlo porque DJF no pertenece al universo mensual que se analizará, por lo que imputarlo no aporta información útil al análisis final.

### 4.2 — Trasladar la limpieza al módulo (0.2 puntos)

Después de probar las expresiones en el notebook, implementen en
`src/meteolab/limpieza.py`:

- `limpiar_temperaturas`, compatible con `DataFrame` y `LazyFrame`;
- `resumen_de_nulos`, para revisar los valores faltantes;
- `claves_repetidas`, para detectar repeticiones de país, año y período.

Las tres funciones se utilizan en las pruebas de esta etapa.

In [ ]:
from src.meteolab.limpieza import limpiar_temperaturas, resumen_de_nulos

limpias_modulo = limpiar_temperaturas(validado)

---
# Etapa 5 — Fechas y agregaciones mensuales (1.1 puntos)

`year` y `period` son columnas separadas. Construirán una fecha para ordenar
las observaciones y luego resumirán los años disponibles por mes.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — columna derivada</strong>
  Una columna derivada se calcula a partir de columnas existentes. La columna <code>fecha</code> no agrega una medición: combina <code>year</code> y el número del mes para permitir ordenamiento y gráficos temporales.
</div>

### 5.1 — Investigar y construir fechas (0.2 puntos)

Investiguen cómo traducir los códigos `JAN` a `DEC` a números de mes y cómo
convertir una cadena con formato ISO en una fecha de Polars. Construyan en el
notebook las columnas `month` y `fecha`, y comprueben que el resultado se
ordena cronológicamente.

In [ ]:
from src.meteolab.constantes import MESES

In [ ]:
mensuales = limpias_modulo.with_columns(
    pl.col("period").replace(MESES).cast(pl.Int64).alias("month")
).with_columns(
    pl.date(
        pl.col("year"),
        pl.col("month"),
        1,
    ).alias("fecha")
)

In [ ]:
mensuales.select(
    "year",
    "period",
    "month",
    "fecha",
).head()

In [ ]:
mensuales.sort("fecha").select(
    "year",
    "period",
    "month",
    "fecha",
).head(15)

### 5.2 — Investigar y calcular resúmenes (0.4 puntos)

El resultado de esta exploración se trasladará después a
`src/meteolab/metricas.py`, en `resumen_mensual`. Debe devolver una fila por
país y mes, con:

- `iso_alpha3` y `country`;
- `month`;
- `observaciones`;
- `temperature_mean`, redondeada a dos decimales.

Antes de escribir la función, investiguen `group_by` y `agg`. Calculen el
resumen directamente en el notebook y revisen algunas filas.

In [ ]:
climatologia = (
    mensuales
    .group_by(["iso_alpha3", "country", "month"])
    .agg(
        pl.len().alias("observaciones"),
        pl.col("temperature_c").mean().round(2).alias("temperature_mean"),
    )
    .sort(["iso_alpha3", "month"])
)

In [ ]:
PAISES = ["CHL", "ARG", "PER", "BOL", "BRA", "CAN", "EGY"]
fig = px.line(
    climatologia.filter(pl.col("iso_alpha3").is_in(PAISES)),
    x="month",
    y="temperature_mean",
    color="country",
    markers=True,
    title="Climatología mensual de países seleccionados",
    labels={
        "month": "Mes",
        "temperature_mean": "Temperatura media (°C)",
        "country": "País",
    },
)
fig.show()

Calculen también una media por año, pero debe salir de las filas mensuales
limpias. No usen las filas `ANN` del CSV. El resumen debe conservar los años
incompletos y la columna `meses_disponibles`; para comparar períodos, filtren
después los años cuyo conteo sea igual a 12.

In [ ]:
anuales_desde_meses = (
    mensuales
    .group_by(["iso_alpha3", "country", "year"])
    .agg(
        pl.col("month").n_unique().alias("meses_disponibles"),
        pl.col("temperature_c").mean().round(2).alias("temperature_mean"),
    )
    .sort(["iso_alpha3", "year"])
)

In [ ]:
anuales_completos = anuales_desde_meses.filter(
    pl.col("meses_disponibles") == 12
)
print("Años completos:", anuales_completos.height)

Revisen si la tabla contiene más de una observación para la misma combinación
de país, año y período. Investiguen `group_by` y `len` para contar posibles
repeticiones de esa clave.

In [ ]:
repetidas = (
    mensuales
    .group_by(["country", "year", "period"])
    .agg(pl.len())
    .filter(pl.col("len") > 1)
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 9 — Qué significa una climatología mensual (0.2 puntos)</strong>
  En `climatologia`, ¿qué representa una fila para `CHL` y `month = 1`? ¿Por qué esa fila resume varios años y no un solo registro del archivo?
</div>

Una fila de climatologia para CHL y month = 1 representa la temperatura media de enero para Chile considerando todos los años disponibles. Resume varios años porque se agruparon todas las observaciones de enero del país y se calculó su promedio, en vez de representar una sola fila original del archivo.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 10 — Detectar duplicados (0.2 puntos)</strong>
  ¿Qué combinación de columnas debería identificar de forma única una observación mensual? ¿Cómo detectarían registros duplicados antes de calcular las métricas?
</div>

Una observación mensual debería quedar identificada de forma única por la combinación de país, año y período (country, year, period). Para detectar duplicados, agruparía por esas columnas, contaría las filas de cada grupo y revisaría las combinaciones cuyo conteo sea mayor que 1 antes de calcular las métricas.


---

### 5.3 — Trasladar las transformaciones al módulo (0.1 puntos)

Después de probar las expresiones, implementen `agregar_fecha_mensual`,
`resumen_mensual` y `resumen_anual_desde_mensuales` en sus módulos. Comparen
los resultados del módulo con los que obtuvieron directamente en el
notebook.

In [ ]:
from src.meteolab.derivadas import agregar_fecha_mensual
from src.meteolab.metricas import (
    resumen_anual_desde_mensuales,
    resumen_mensual,
)
from src.meteolab.limpieza import claves_repetidas

In [ ]:
mensuales_modulo = agregar_fecha_mensual(limpias_modulo)
climatologia_modulo = resumen_mensual(mensuales_modulo)
anuales_modulo = resumen_anual_desde_mensuales(mensuales_modulo, ["CHL"])
repetidas_modulo = claves_repetidas(mensuales_modulo)

---
# Etapa 6 — Ventanas y anomalías mensuales (0.8 puntos)

Una media histórica de enero no debe compararse con una de julio. Usarán una
ventana por país y mes para medir cuánto se aparta cada observación de sus
pares comparables.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — ventana</strong>
  Una expresión con <code>.over("iso_alpha3", "month")</code> calcula una medida dentro de cada grupo y devuelve ese resultado en las filas originales. A diferencia de <code>group_by().agg()</code>, una ventana conserva una fila por observación.
</div>

### 6.1 — Investigar y calcular una ventana (0.4 puntos)

Investiguen `over` y construyan directamente en el notebook las tres
columnas solicitadas. Usen la dupla `(iso_alpha3, month)` como grupo de
comparación para que cada mes se compare con otros del mismo país.

Implementen `anomalias_mensuales(mensuales, umbral=2.0)` en el módulo después
de comprobar el resultado de esta exploración. Debe agregar:

- `temperature_mean_month`, la media histórica del país para ese mes;
- `standardized_anomaly`, la diferencia dividida por la desviación estándar;
- `is_anomaly`, booleana y sin nulos.

In [ ]:
marcadas = mensuales.with_columns(
    pl.col("temperature_c")
    .mean()
    .over(["iso_alpha3", "month"])
    .alias("temperature_mean_month")
).with_columns(
    pl.when(
        pl.col("temperature_c")
        .std()
        .over(["iso_alpha3", "month"]) > 0
    )
    .then(
        (pl.col("temperature_c") - pl.col("temperature_mean_month"))
        / pl.col("temperature_c").std().over(["iso_alpha3", "month"])
    )
    .otherwise(0.0)
    .fill_null(0.0)
    .alias("standardized_anomaly")
).with_columns(
    (pl.col("standardized_anomaly").abs() > 2.0)
    .alias("is_anomaly")
)

In [ ]:
fig = px.scatter(
    marcadas.filter(pl.col("iso_alpha3").is_in(PAISES)),
    x="fecha",
    y="standardized_anomaly",
    color="is_anomaly",
    facet_row="iso_alpha3",
    hover_data=["country", "period", "temperature_c"],
    title="Anomalías mensuales",
    labels={
        "fecha": "Fecha",
        "standardized_anomaly": "Anomalía estandarizada",
        "is_anomaly": "¿Anómala?",
    },
)
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 11 — Elegir el grupo de comparación (0.3 puntos)</strong>
  ¿Qué diferencia habría entre calcular la anomalía sobre `iso_alpha3` y calcularla sobre `(iso_alpha3, month)`? ¿Cuál de las dos opciones permite distinguir mejor una variación inusual de una diferencia normal entre estaciones del año?
</div>

Calcular la anomalía solo sobre `iso_alpha3` compararía cada observación con todas las temperaturas históricas del país, mezclando meses fríos y cálidos. Esto puede hacer que una diferencia normal entre estaciones parezca una anomalía.

En cambio, calcularla sobre `(iso_alpha3, month)` compara cada observación con el mismo mes del mismo país. Por ejemplo, enero de Chile se compara con otros eneros de Chile.

Por eso, usar `(iso_alpha3, month)` permite distinguir mejor una variación realmente inusual de una diferencia normal entre estaciones del año, porque controla la estacionalidad.

---

### 6.2 — Trasladar la ventana al módulo (0.1 puntos)

Después de revisar el resultado, implementen `anomalias_mensuales` en
`src/meteolab/metricas.py`. Comparen las columnas y la cantidad de filas con
el cálculo realizado directamente en el notebook.

In [ ]:
from src.meteolab.metricas import anomalias_mensuales

marcadas_modulo = anomalias_mensuales(mensuales_modulo, umbral=2.0)

---
# Etapa 7 — Pipeline lazy y análisis (1.1 puntos)

Integrarán las funciones en una consulta lazy. El pipeline debe leer el CSV,
descartar los períodos no mensuales, construir las fechas y producir el
resumen pedido sin materializar pasos intermedios.

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Contrato del pipeline</strong>
  <code>pipeline_mensual</code> debe devolver un <code>LazyFrame</code>. El filtrado de países, la selección de meses y la construcción de columnas deben formar parte del plan. La ejecución ocurre solo con <code>collect()</code>.
</div>

### 7.1 — Investigar y construir una consulta lazy (0.1 puntos)

Investiguen cómo combinar `scan_csv`, `filter`, `with_columns` y `collect`.
Construyan primero una consulta lazy directamente en el notebook que:

- lea el CSV con los tipos esperados;
- conserve solo las filas mensuales con temperatura disponible;
- seleccione los países de `PAISES`;
- agregue `month` y `fecha`;
- ordene por país y fecha.

Comprueben que la consulta no se ejecuta hasta llamar a `collect()` y revisen
las primeras filas del resultado.

In [ ]:
from src.meteolab.constantes import ESQUEMA_CRU

In [ ]:
consulta_notebook = (
    pl.scan_csv(
        RUTA_CSV,
        schema_overrides=ESQUEMA_CRU,
    )
    .filter(
        pl.col("period").is_in(PERIODOS_MENSUALES)
        & pl.col("temperature_c").is_not_null()
    )
    .filter(
        pl.col("iso_alpha3").is_in(PAISES)
    )
    .with_columns(
        pl.col("period")
        .replace(MESES)
        .cast(pl.Int8)
        .alias("month")
    )
    .with_columns(
        pl.date(
            pl.col("year"),
            pl.col("month"),
            1,
        ).alias("fecha")
    )
    .sort(["country", "fecha"])
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 12 — Elegir entre lazy y eager (0.2 puntos)</strong>
  Comparen esta consulta lazy con una solución eager que lea el archivo mediante `read_csv` y ejecute cada transformación de inmediato. ¿Qué ventajas ofrece construir un `LazyFrame` y ejecutar al final con `collect()`? Mencionen dos ventajas concretas y una situación en que preferirían trabajar en modo eager.
</div>

Trabajar con un `LazyFrame` permite definir todas las transformaciones antes de ejecutarlas. Una ventaja es que Polars puede optimizar el plan de consulta, por ejemplo aplicando filtros antes de realizar operaciones innecesarias. Otra ventaja es que evita materializar tablas intermedias, lo que puede reducir el uso de memoria.

La ejecución se realiza recién al llamar a `collect()`.

Preferiría trabajar en modo eager cuando el dataset es pequeño y necesito inspeccionar inmediatamente los resultados de cada transformación durante una exploración o depuración.

### 7.2 — Trasladar el pipeline al módulo (0.2 puntos)

Después de probar la consulta, implementen en `src/meteolab/reporte.py`:

- `pipeline_mensual`;
- `pipeline_resumen_mensual`;
- `pipeline_resumen_anual`, calculado desde meses;
- `pipeline_anomalias`;
- `ejecutar_reporte` y `plan_de_ejecucion`.

In [ ]:
from src.meteolab.reporte import (
    ejecutar_reporte,
    pipeline_anomalias,
    pipeline_mensual,
    pipeline_resumen_anual,
    pipeline_resumen_mensual,
    plan_de_ejecucion,
)

consulta = pipeline_mensual(RUTA_CSV, PAISES)
print(type(consulta))
print(plan_de_ejecucion(RUTA_CSV, PAISES))
print("Filas directas:", consulta_notebook.collect().height)
print("Filas del módulo:", consulta.collect().height)

In [ ]:
reporte_mensual = ejecutar_reporte(RUTA_CSV, PAISES)
print(reporte_mensual.shape)
display(reporte_mensual.head())

In [ ]:
reporte_anual = pipeline_resumen_anual(RUTA_CSV, PAISES).collect()
anomalias = pipeline_anomalias(RUTA_CSV, ["CHL"]).collect()
print("Años calculados desde meses:", reporte_anual.height)
print("Anomalías marcadas:", anomalias["is_anomaly"].sum())

### 7.3 — Analizar cambios de temperatura a largo plazo (0.2 puntos)

Construyan primero una serie con la media anual de cada país y grafiquen cómo
evoluciona desde 1901. Después comparen la media de 1901–1930 con la de
1991–2020. Estas comparaciones describen cambios de temperatura; por sí solas
no demuestran sus causas.

In [ ]:
evolucion_anual = reporte_anual.filter(
    pl.col("meses_disponibles") == 12
)

In [ ]:
fig = px.line(
    evolucion_anual,
    x="year",
    y="temperature_mean",
    color="country",
    title="Evolución de la temperatura media anual",
    labels={
        "year": "Año",
        "temperature_mean": "Temperatura media (°C)",
        "country": "País",
    },
)
fig.show()

Comparen ahora los dos períodos de referencia para resumir el cambio.

In [ ]:
periodo_inicial = (
    evolucion_anual
    .filter(
        pl.col("year").is_between(1901, 1930)
    )
    .group_by("country")
    .agg(
        pl.col("temperature_mean")
        .mean()
        .round(2)
        .alias("media_1901_1930")
    )
)

periodo_final = (
    evolucion_anual
    .filter(
        pl.col("year").is_between(1991, 2020)
    )
    .group_by("country")
    .agg(
        pl.col("temperature_mean")
        .mean()
        .round(2)
        .alias("media_1991_2020")
    )
)

comparacion = (
    periodo_inicial
    .join(periodo_final, on="country")
    .with_columns(
        (
            pl.col("media_1991_2020")
            - pl.col("media_1901_1930")
        )
        .round(2)
        .alias("cambio_c")
    )
)

In [ ]:
fig = px.bar(
    comparacion,
    x="country",
    y="cambio_c",
    color="cambio_c",
    title="Cambio de temperatura media entre períodos de referencia",
    labels={
        "country": "País",
        "cambio_c": "Cambio de temperatura (°C)",
    },
)
fig.update_xaxes(categoryorder="total descending")
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 13 — Cambios de temperatura y sus límites (0.2 puntos)</strong>
  Observen la evolución anual y la tabla <code>comparacion</code>. ¿Qué países presentan el mayor aumento entre los dos períodos? ¿Todos muestran el mismo cambio? Usen valores concretos para describir la tendencia y expliquen por qué este análisis aporta evidencia descriptiva para estudiar el calentamiento global, pero no permite atribuir causas ni calcular por sí solo una temperatura global.
</div>

Los países no muestran el mismo cambio de temperatura entre ambos períodos. Canadá presenta el mayor aumento, con aproximadamente **1,28 °C**, seguido por Egipto con cerca de **0,92 °C** y Brasil con **0,83 °C**. En cambio, Bolivia presenta el menor aumento, cercano a **0,08 °C**. Argentina aumenta aproximadamente **0,47 °C**, Perú **0,36 °C** y Chile **0,29 °C**.

En general, el gráfico de evolución anual muestra una tendencia de aumento de la temperatura media, aunque la magnitud del cambio varía entre países.

Este análisis aporta evidencia descriptiva para estudiar el calentamiento global, porque permite observar que las temperaturas medias recientes son mayores que las del período 1901–1930 en todos los países analizados. Sin embargo, no permite atribuir las causas de esos cambios, ya que solo describe la evolución de las temperaturas observadas. Tampoco permite calcular por sí solo una temperatura global, porque el análisis considera un conjunto específico de países y no realiza una ponderación que represente a todo el planeta.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 14 — Interpretar el resultado (0.2 puntos)</strong>
  Elijan un país y describan dos patrones que observen en su climatología mensual o en sus anomalías. Usen valores concretos de las tablas o gráficos. Indiquen también qué información no se puede concluir a partir de este archivo.
</div>

Al comparar Chile y Brasil se observan dos patrones principales.

Primero, **Chile presenta una estacionalidad mucho más marcada**. Su temperatura media mensual es de aproximadamente **13 °C en enero**, disminuye hasta cerca de **5–6 °C en junio-julio** y luego vuelve a aumentar hacia diciembre. En cambio, **Brasil mantiene temperaturas mucho más estables durante el año**, alrededor de **23–26 °C**, con una disminución relativamente pequeña durante los meses de invierno.

Segundo, en ambos países aparecen **anomalías mensuales a lo largo de la serie histórica**, es decir, meses cuya temperatura se aleja considerablemente de lo habitual para ese mismo país y mes. Estas anomalías no aparecen únicamente en una época específica, sino que se observan en distintos años de la serie.

Por lo tanto, Chile y Brasil tienen comportamientos estacionales distintos: Chile presenta una variación anual mucho mayor, mientras que Brasil mantiene temperaturas altas y relativamente estables durante todo el año.

A partir de este archivo podemos describir patrones y cambios de temperatura, pero **no podemos determinar sus causas** ni atribuir las anomalías a fenómenos específicos. Tampoco podemos concluir por estos datos, por sí solos, que un evento particular se deba al cambio climático.

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Antes de entregar</strong>
  Ejecuten las pruebas por etapa y luego la suite completa:

  <pre><code>uv run pytest -m etapa1
uv run pytest -m etapa2
uv run pytest -m etapa3
uv run pytest -m etapa4
uv run pytest -m etapa5
uv run pytest -m etapa6
uv run pytest -m etapa7
uv run pytest</code></pre>

  Reinicien el kernel y ejecuten el notebook completo. Revisen que los gráficos Plotly se rendericen y que las respuestas usen resultados observables.
</div>